# Europarl scaled v3: exploratory data analysis

This notebook reads the frozen document-stratified Europarl scaled v3 split and saved EDA outputs. No model is trained here. This is the controlled scaled v3 subset: 12,000 train, 1,500 validation and 1,500 test pairs across distinct documents. The smaller v2 and pilot results are not loaded here.


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display, Image

START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (directory for directory in (START, *START.parents)
     if (directory / 'data/processed/europarl_scaled_v3').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Europarl scaled v3 data not found above {START}')
DATA = PROJECT_ROOT / 'data/processed/europarl_scaled_v3'
METRICS = PROJECT_ROOT / 'outputs/metrics/europarl_scaled_v3'
FIGURES = PROJECT_ROOT / 'outputs/figures/eda_scaled_v3'
assert DATA.is_dir(), f'Missing {DATA}'
manifest = json.loads((DATA / 'summary.json').read_text(encoding='utf-8'))
frames = []
for split in ('train', 'validation', 'test'):
    path = DATA / f'{split}.csv'
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    assert actual == manifest['csv_sha256'][split], f'Hash mismatch: {path}'
    frame = pd.read_csv(path, dtype={'src_sent_id': str, 'tgt_sent_id': str})
    assert frame['split'].eq(split).all()
    frames.append(frame)
data = pd.concat(frames, ignore_index=True)
print('Verified dataset SHA256 for train, validation and test.')


## Split size and document coverage

Splits are grouped by English source document. Inspect document counts as well as sentence counts; many sentences from one document do not represent independent documents.


In [ ]:
counts = data.groupby(['split', 'pair']).agg(
    sentences=('src', 'size'), documents=('doc_id', 'nunique')
).reset_index()
display(counts)
docs = {split: set(data.loc[data.split == split, 'doc_id'])
        for split in ('train', 'validation', 'test')}
for left, right in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
    assert not (docs[left] & docs[right]), f'Document leakage: {left}/{right}'
print('Split document counts:', {k: len(v) for k, v in docs.items()})
print('Document-disjoint splits: PASS')


## Length and truncation

The model translates the current English source sentence. The context configuration appends the preceding English sentence as `src </s> previous_src` and truncates the source on the right at 128 mBART tokens. These tables were generated separately from the v3 dataset.


In [ ]:
for filename in ('eda_length_summary.csv', 'eda_truncation_summary.csv',
                 'eda_longest_context_examples.csv'):
    path = METRICS / filename
    if path.exists():
        table = pd.read_csv(path)
        print(filename, len(table), 'rows')
        if filename == 'eda_length_summary.csv':
            display(table[(table['split'] == 'all') & (table['pair'] == 'all')])
        elif filename == 'eda_truncation_summary.csv':
            display(table[(table['split'] == 'all') & (table['pair'] == 'all')])
        else:
            display(table.head(5))
    else:
        print('Pending saved EDA table:', path)


## Figures

Saved plots correspond to the multidoc v3 dataset and should not be mixed with v2 or pilot figures.


In [ ]:
for filename in ('01_pair_counts.png', '02_token_length_distributions.png',
                 '03_context_length_boxplot.png', '04_source_target_tokens.png',
                 '05_context_truncation_rate.png'):
    path = FIGURES / filename
    print(filename)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Pending saved figure:', path)


## Modeling note

The 8,000/4,000 English–German/English–French training split motivates pair-balanced loss. The split contains 400 train documents, 100 validation documents and 100 held-out test documents. Analyses that compare systems should report language-pair scores and treat document clustering as a limitation.
